In [1]:
import pandas as pd

### UNION DE LOS DOS DATASETS EN UNO SOLO ###

Primero, cargamos los dos dataframes y hacemos que el ID deje de empezar por PP en el dataframe con los datos clínicos

In [2]:
dataframe_clinicos = pd.read_csv("dataframe_prueba.csv", index_col=0)
dataframe_SAA = pd.read_csv("SAA_Internal_20240227 (1).csv")

dataframe_clinicos["participant_id"] = dataframe_clinicos["participant_id"].str.replace("PP-", "", regex=False)

dataframe_clinicos[dataframe_clinicos["visit_name"] == "BL"]

,participant_id,GUID,visit_name,visit_month,age_at_baseline,sex,ethnicity,race,prodromal_category,study_arm,...,pd_diagnosis_months_after_baseline,age_at_diagnosis,pd_medication_initiation_months_after_baseline,pd_medication_start_months_after_baseline,use_of_pd_medication,pd_medication_recent_use_months_after_baseline,on_levodopa,on_dopamine_agonist,on_other_pd_medications,diagnosis_type


Cambiamos el formato del nombre de las visitas de los datos clínicos para que coincida con el dataframe de SAA

In [13]:
visitname_to_v = {
    "M0": "V00", "M3": "V01",   "M6": "V02",   "M9": "V03",   "M12": "V04",
    "M18": "V05",  "M24": "V06",  "M30": "V07",  "M36": "V08",
    "M42": "V09",  "M48": "V10",  "M54": "V11",  "M60": "V12",
    "M72": "V13",  "M84": "V14",  "M96": "V15",  "M108": "V16",
    "M120": "V17", "M132": "V18", "M144": "V19", "M156": "V20",
    "M168": "V21", "M180": "V22", "M192": "V23", "M204": "V24"
}

# SC -> BL
dataframe_clinicos.loc[dataframe_clinicos["visit_name"] == "SC", "visit_name"] = "BL"

# extraer Mxx (M48, M48#2, etc.)
mask_m = dataframe_clinicos["visit_name"].str.startswith("M", na=False)
dataframe_clinicos.loc[mask_m, "visit_name"] = (
    dataframe_clinicos.loc[mask_m, "visit_name"]
    .str.extract(r"(M\d+)")[0]
    .map(visitname_to_v)
)

dataframe_clinicos[dataframe_clinicos["participant_id"] == "51971"]
dataframe_clinicos.to_csv("dataframe_clinicos_BL.csv")


KeyError: 'visit_name'

Renombramos las columnas por las que vamos a hacer el merge al nombre que tienen en SAA para que no se dupliquen

In [23]:
dataframe_clinicos = dataframe_clinicos.rename(columns={"participant_id": "PATNO", "visit_name": "CLINICAL_EVENT"})

dataframe_clinicos.to_csv("dataframe_clinicos_BL.csv")


Hacemos el merge de ambos dataframes

In [46]:
dataframe_clinicos["PATNO"] = pd.to_numeric(dataframe_clinicos["PATNO"], errors="coerce")
dataframe_SAA["PATNO"] = pd.to_numeric(dataframe_SAA["PATNO"], errors="coerce")


dataframe_SAA_clinico = dataframe_clinicos.merge(
    dataframe_SAA,
    on=["PATNO", "CLINICAL_EVENT"],
    how= "left"
)


dataframe_SAA_clinico.to_csv("dataframe_SAA_clinico_V0.csv")

dataframe_SAA_clinico

,PATNO,GUID,CLINICAL_EVENT,visit_month,age_at_baseline,sex,ethnicity,race,prodromal_category,study_arm,...,InstrumentRep1,InstrumentRep2,InstrumentRep3,SampleVolRep1,SampleVolRep2,SampleVolRep3,RUNDATE,PROJECTID,PI_NAME,PI_INSTITUTION
0,3433,NIHAA107VTGPE,BL,-1.0,82.0,Female,Not Hispanic or Latino,White,Unknown/Not collected as enrollment criterion,PD,...,6.0,6.0,6.0,NaN,NaN,NaN,2022-05-12,155.0,Claudia Soto,Amprion
1,3433,NIHAA107VTGPE,V01,3.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3433,NIHAA107VTGPE,V02,6.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3433,NIHAA107VTGPE,V03,9.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3433,NIHAA107VTGPE,V04,12.0,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,2.0,2.0,NaN,NaN,NaN,2023-12-18,237.0,Luis Concha,Amprion
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14695,92834,NaN,V08,36.0,NaN,NaN,NaN,NaN,NaN,NaN,...,6.0,6.0,6.0,NaN,NaN,NaN,2022-03-31,155.0,Claudia Soto,Amprion
14696,92834,NaN,V09,42.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14697,92834,NaN,V10,48.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14698,92834,NaN,V11,54.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1]:
dataframe_SAA_clinico[dataframe_SAA_clinico["SEX"]]

NameError: name 'dataframe_SAA_clinico' is not defined